# Emulators

ProbPipe provides an **emulator** abstraction for probabilistic surrogate models.
An emulator wraps any model that maps inputs to a *predictive distribution*
over outputs. The design follows TFP-style shape semantics
(`sample_shape + batch_shape + event_shape`) and supports four "joint modes"
that control which axes are modelled jointly (event) versus independently (batch).

## Shape Semantics

Given input `X` with shape `(*extra_batch, n, *input_shape)`, the returned
distribution's `batch_shape` and `event_shape` depend on two boolean flags:

| `joint_inputs` | `joint_outputs` | `event_shape`         | `batch_shape`                    |
|:--------------:|:---------------:|:---------------------:|:--------------------------------:|
| False          | False           | `()`                  | `(*extra_batch, n, *output_shape)` |
| True           | False           | `(n,)`                | `(*extra_batch, *output_shape)`  |
| False          | True            | `(*output_shape,)`    | `(*extra_batch, n)`              |
| True           | True            | `(n, *output_shape)`  | `(*extra_batch,)`                |

In every mode a *sample* has the same total shape:
`(*sample_shape, *extra_batch, n, *output_shape)`.
The flags only change which axes are jointly modelled (event) vs independent (batch).

## Class Hierarchy

```
Emulator (abstract)
  +-- GaussianEmulator (abstract: predict_mean, predict_variance)
        +-- LinearGaussianRegressor   (f(x) = bias + Phi(x) @ w)
        +-- LinCombGaussianWeights    (f(x) = bias + Phi @ w(x))
        +-- (your custom GP, synthetic-likelihood emulator, ...)
```

This notebook walks through each of these and shows how to build your own.

In [1]:
import jax
import jax.numpy as jnp

from probpipe import (
    Emulator,
    GaussianEmulator,
    LinearGaussianRegressor,
    LinCombGaussianWeights,
    MultivariateNormal,
    Normal,
)

key = jax.random.PRNGKey(0)

## 1. Custom GaussianEmulator -- a Toy GP with an RBF Kernel

The simplest way to create an emulator is to subclass `GaussianEmulator` and
implement `predict_mean`, `predict_variance`, and (optionally)
`predict_covariance`.

Below we build a zero-mean GP prior with a squared-exponential (RBF) kernel.
Because the GP provides a full covariance matrix across input points, we set
`supports_joint_inputs = True`.

In [2]:
class ToyGP(GaussianEmulator):
    """Zero-mean GP with an RBF kernel."""

    supports_joint_inputs = True

    def __init__(self, lengthscale=1.0, variance=1.0, noise=0.01):
        super().__init__(input_shape=(1,), output_shape=())
        self._ls = lengthscale
        self._var = variance
        self._noise = noise

    def predict_mean(self, X):
        extra_batch, n = self._parse_X(X)
        return jnp.zeros((*extra_batch, n))

    def predict_variance(self, X):
        extra_batch, n = self._parse_X(X)
        return jnp.full((*extra_batch, n), self._var)

    def predict_covariance(self, X, *, joint_inputs=False, joint_outputs=False):
        if not joint_inputs:
            raise NotImplementedError
        # Squared-exponential kernel
        sq_dist = jnp.sum((X[..., :, None, :] - X[..., None, :, :]) ** 2, axis=-1)
        K = self._var * jnp.exp(-0.5 * sq_dist / self._ls ** 2)
        return K + self._noise * jnp.eye(K.shape[-1])


gp = ToyGP(lengthscale=0.5, variance=1.0, noise=0.01)
print(gp)

ToyGP(input_shape=(1,), output_shape=())


### 1.1 Prediction Inputs

We create a small set of test inputs. Because `input_shape=(1,)`, each point
is a 1-D vector, so `X` has shape `(n, 1)`.

In [22]:
X = jnp.linspace(-2, 2, 6).reshape(-1, 1)  # (6, 1)
print(f"X.shape: {X.shape}")

X.shape: (6, 1)


### 1.2 The Four Joint Modes

We call the emulator in each of the four joint modes and inspect the
resulting distribution's `batch_shape` and `event_shape`.

**Mode 1: `joint_inputs=False, joint_outputs=False` (default)**

Each input point gets an independent scalar prediction. The `n` dimension
and any `output_shape` dimensions are all batch.

In [24]:
dist_marginal = gp(X)  # default: joint_inputs=False, joint_outputs=False
print("Mode: joint_inputs=False, joint_outputs=False")
print(f"  gp(X): {dist_marginal}")
print(f"  batch_shape: {dist_marginal.batch_shape}")
print(f"  event_shape: {dist_marginal.event_shape}")
print(f"  mean: {dist_marginal.mean()}")
print(f"  variance: {dist_marginal.variance()}")

Mode: joint_inputs=False, joint_outputs=False
  gp(X): Normal(event_shape=(), batch_shape=(6,))
  batch_shape: (6,)
  event_shape: ()
  mean: [0. 0. 0. 0. 0. 0.]
  variance: [1. 1. 1. 1. 1. 1.]


**Mode 2: `joint_inputs=True, joint_outputs=False`**

Predictions are correlated across the `n` input points. The returned
distribution is a `MultivariateNormal` with `event_shape=(n,)`.

In [25]:
dist_joint_in = gp(X, joint_inputs=True)
print("Mode: joint_inputs=True, joint_outputs=False")
print(f"  gp(X): {dist_joint_in}")
print(f"  batch_shape: {dist_joint_in.batch_shape}")
print(f"  event_shape: {dist_joint_in.event_shape}")
print(f"  mean shape: {dist_joint_in.mean().shape}")
print(f"  cov shape:  {dist_joint_in.cov.shape}")

Mode: joint_inputs=True, joint_outputs=False
  gp(X): MultivariateNormal(event_shape=(6,))
  batch_shape: ()
  event_shape: (6,)
  mean shape: (6,)
  cov shape:  (6, 6)


Since our `ToyGP` has scalar `output_shape=()`, the remaining two modes
(`joint_outputs=True`) would require a multi-output emulator. We will
demonstrate those modes later with `LinearGaussianRegressor`.

### 1.3 Extra Batch Dimensions

Any leading dimensions beyond `(n, *input_shape)` are treated as extra
batch dimensions and passed through to the output distribution.

In [28]:
# Add an extra batch dimension of size 3
X_batched = jnp.stack([X, X + 1.0, X + 2.0])  # (3, 6, 1)
print(f"X_batched.shape: {X_batched.shape}")

dist_batch_marginal = gp(X_batched)
print(f"\nMarginal mode with extra batch:")
print(f"gp(X_batched): {dist_batch_marginal}")

dist_batch_joint = gp(X_batched, joint_inputs=True)
print(f"\nJoint-input mode with extra batch:")
print(f"gp(X_batched): {dist_batch_joint}")

X_batched.shape: (3, 6, 1)

Marginal mode with extra batch:
gp(X_batched): Normal(event_shape=(), batch_shape=(3, 6))

Joint-input mode with extra batch:
gp(X_batched): MultivariateNormal(event_shape=(6,), batch_shape=(3,))


### 1.4 Sampling

Draw samples from the joint distribution and inspect shapes.

In [ ]:
k1, k2 = jax.random.split(key)

# Marginal samples: each point is independent
samples_marginal = dist_marginal.sample(k1, (4,))
print(f"Marginal samples shape: {samples_marginal.shape}")
print(f"  (sample_shape=4, batch_shape=n=6)")

# Joint samples: correlated across input points
samples_joint = dist_joint_in.sample(k2, (4,))
print(f"\nJoint samples shape: {samples_joint.shape}")
print(f"  (sample_shape=4, event_shape=n=6)")
print(f"\nOne joint sample (a correlated draw across 6 points):")
print(f"  {samples_joint[0]}")

## 2. Wrapping a GPJax Model

In practice you may train a GP using a library like
[GPJax](https://github.com/JaxGaussianProcesses/GPJax) and then want to
plug it into a ProbPipe pipeline. The pattern is the same: subclass
`GaussianEmulator` and implement the three prediction methods.

The code skeleton below is **illustrative** -- it assumes a trained GPJax
posterior object and requires `gpjax` to be installed.

```python
import gpjax
from probpipe import GaussianEmulator


class GPJaxEmulator(GaussianEmulator):
    """Wrap a trained GPJax posterior as a ProbPipe GaussianEmulator."""

    supports_joint_inputs = True

    def __init__(self, posterior, train_data, input_dim):
        super().__init__(input_shape=(input_dim,), output_shape=())
        self._posterior = posterior
        self._train_data = train_data

    def predict_mean(self, X):
        # GPJax latent_predict returns (mean, var)
        latent_dist = self._posterior.predict(X, train_data=self._train_data)
        return latent_dist.mean()

    def predict_variance(self, X):
        latent_dist = self._posterior.predict(X, train_data=self._train_data)
        return latent_dist.variance()

    def predict_covariance(self, X, *, joint_inputs=False, joint_outputs=False):
        if not joint_inputs:
            raise NotImplementedError
        latent_dist = self._posterior.predict(X, train_data=self._train_data)
        return latent_dist.covariance()
```

Once wrapped, the emulator can be called exactly like any other ProbPipe
emulator:

```python
emu = GPJaxEmulator(posterior, train_data, input_dim=2)
dist = emu(X_test, joint_inputs=True)  # MultivariateNormal
```

## 3. Synthetic Likelihood Emulator

The `GaussianEmulator` base class is not limited to GP surrogates. Any
model that produces Gaussian (or Gaussian-approximated) predictions can
use it. A compelling example is **synthetic likelihood** (Wood, 2010),
where the likelihood at each parameter value `theta` is approximated by
a Gaussian fitted to repeated simulator runs.

Key difference from a GP: predictions at different input points are
*independent* (there is no cross-input covariance), so we set
`supports_joint_inputs = False`.

### 3.1 A Toy Stochastic Model

Consider the model `y = theta^2 + noise` where `noise ~ N(0, 0.1)`.
The synthetic likelihood at `theta` is `N(theta^2, 0.1)`. We encode
this directly as an emulator.

In [29]:
class SyntheticLikelihoodEmulator(GaussianEmulator):
    """Emulator where each input gets an independent Gaussian likelihood.

    Models y = theta^2 + noise, noise ~ N(0, 0.1).
    The 'emulated' likelihood at theta is N(theta^2, 0.1).
    """

    supports_joint_inputs = False  # predictions are independent across theta

    def __init__(self):
        super().__init__(input_shape=(1,), output_shape=())

    def predict_mean(self, X):
        # Likelihood mean is a deterministic function of theta
        return jnp.squeeze(X ** 2, axis=-1)

    def predict_variance(self, X):
        extra_batch, n = self._parse_X(X)
        return jnp.full((*extra_batch, n), 0.1)


sl = SyntheticLikelihoodEmulator()
print(sl)
print(f"supports_joint_inputs: {sl.supports_joint_inputs}")

SyntheticLikelihoodEmulator(input_shape=(1,), output_shape=())
supports_joint_inputs: False


### 3.2 Calling the Synthetic Likelihood Emulator

In [30]:
theta = jnp.array([[0.5], [1.0], [1.5], [2.0]])  # (4, 1)
dist_sl = sl(theta)

print(f"Input shape: {theta.shape}")
print(f"batch_shape: {dist_sl.batch_shape}")
print(f"event_shape: {dist_sl.event_shape}")
print(f"\nPredictive means (theta^2): {dist_sl.mean()}")
print(f"Predictive variances:        {dist_sl.variance()}")

Input shape: (4, 1)
batch_shape: (4,)
event_shape: ()

Predictive means (theta^2): [0.25 1.   2.25 4.  ]
Predictive variances:        [0.1 0.1 0.1 0.1]


In [ ]:
# joint_inputs=True is NOT supported -- verify the error
try:
    sl(theta, joint_inputs=True)
except ValueError as e:
    print(f"Expected error: {e}")

### 3.3 Log-Probability Evaluation

Since the returned distribution is a standard ProbPipe `Normal`, we can
evaluate log-probabilities at observed data. This is the synthetic
log-likelihood at each theta value.

In [31]:
y_obs = jnp.array([0.3, 1.1, 2.3, 3.9])  # observed data
log_lik = dist_sl.log_prob(y_obs)
print(f"Log-likelihood at each theta: {log_lik}")
print(f"Shape: {log_lik.shape}")

Log-likelihood at each theta: [0.21985404 0.182354   0.21985403 0.182354  ]
Shape: (4,)


This demonstrates the flexibility of the `GaussianEmulator` abstraction:
the same interface handles GP surrogates (with correlated predictions)
and synthetic likelihoods (with independent predictions). Downstream
consumers of the emulator do not need to know which case they are in.

## 4. LinearGaussianRegressor

The `LinearGaussianRegressor` models:

$$f(x) = \text{bias} + \Phi(x)\, w, \qquad w \sim \mathcal{N}(m, C)$$

where $\Phi(x)$ is a user-supplied feature map and $w$ is a fixed Gaussian
weight distribution. The predictive distribution is analytically Gaussian:

- Mean: $\Phi(x)\, m$
- Covariance: $\Phi(x)\, C\, \Phi(x)^T$

This emulator always supports `joint_inputs=True` because the cross-input
covariance is available in closed form.

### 4.1 Polynomial Feature Map

We define a cubic polynomial basis: $\Phi(x) = [1, x, x^2, x^3]$.

In [32]:
def poly_features(X):
    """Cubic polynomial features: [1, x, x^2, x^3].

    Parameters
    ----------
    X : array, shape (*extra_batch, n, 1)

    Returns
    -------
    array, shape (*extra_batch, n, 4)
    """
    x = X[..., 0]  # (*extra_batch, n)
    return jnp.stack([jnp.ones_like(x), x, x ** 2, x ** 3], axis=-1)


# Verify feature map shape
X_test = jnp.linspace(-1, 1, 5).reshape(-1, 1)  # (5, 1)
phi = poly_features(X_test)
print(f"X_test.shape: {X_test.shape}")
print(f"phi.shape:    {phi.shape}")
print(f"phi[0]:       {phi[0]}  # [1, x, x^2, x^3] at x=-1")

X_test.shape: (5, 1)
phi.shape:    (5, 4)
phi[0]:       [ 1. -1.  1. -1.]  # [1, x, x^2, x^3] at x=-1


### 4.2 Weight Distribution

We create a `MultivariateNormal` prior over the 4 polynomial coefficients.

In [33]:
w_mean = jnp.array([1.0, 0.5, -0.3, 0.1])  # prior mean for [c0, c1, c2, c3]
w_cov = 0.1 * jnp.eye(4)  # prior covariance (isotropic)

weights = MultivariateNormal(loc=w_mean, cov=w_cov)
print(f"Weight distribution: {weights}")
print(f"  event_shape: {weights.event_shape}")
print(f"  mean: {weights.mean()}")

Weight distribution: MultivariateNormal(event_shape=(4,))
  event_shape: (4,)
  mean: [ 1.   0.5 -0.3  0.1]


### 4.3 Creating the Regressor

In [34]:
regressor = LinearGaussianRegressor(
    feature_map=poly_features,
    weights=weights,
    input_shape=(1,),
    output_shape=(),
)
print(regressor)
print(f"supports_joint_inputs: {regressor.supports_joint_inputs}")

LinearGaussianRegressor(input_shape=(1,), output_shape=())
supports_joint_inputs: True


### 4.4 Predictions

Call the regressor in both marginal and joint mode.

In [36]:
X_pred = jnp.linspace(-2, 2, 8).reshape(-1, 1)  # (8, 1)

# Marginal predictions
dist_m = regressor(X_pred)
print("Marginal predictions:")
print(f"  regressor(X_pred): {dist_m}")
print(f"  batch_shape: {dist_m.batch_shape}, event_shape: {dist_m.event_shape}")
print(f"  means: {dist_m.mean()}")
print(f"  std:   {jnp.sqrt(dist_m.variance())}")

# Joint predictions across inputs
dist_j = regressor(X_pred, joint_inputs=True)
print(f"\nJoint predictions:")
print(f"  regressor(X_pred): {dist_j}")
print(f"  batch_shape: {dist_j.batch_shape}, event_shape: {dist_j.event_shape}")
print(f"  cov shape: {dist_j.cov.shape}")

Marginal predictions:
  regressor(X_pred): Normal(event_shape=(), batch_shape=(8,))
  batch_shape: (8,), event_shape: ()
  means: [-2.        -0.6180759  0.2880466  0.8303208  1.1206998  1.271137
  1.3935862  1.5999999]
  std:   [2.9154758  1.2532202  0.5168209  0.3299758  0.32997584 0.5168209
 1.2532207  2.9154758 ]

Joint predictions:
  regressor(X_pred): MultivariateNormal(event_shape=(8,))
  batch_shape: (), event_shape: (8,)
  cov shape: (8, 8)


## 5. LinCombGaussianWeights

`LinCombGaussianWeights` transforms a **weight emulator** (mapping inputs
to a Gaussian over a weight vector) into a higher-dimensional output space
via a fixed linear map:

$$f(x) = a + \Phi\, w(x)$$

where $w(x)$ is itself a `GaussianEmulator` and $\Phi$ is a fixed matrix.

This is useful when the latent model operates in a low-dimensional weight
space but the observations live in a higher-dimensional space.

### 5.1 Setting Up

We use the `LinearGaussianRegressor` from above as a weight emulator.
Its `output_shape` is `()` (scalar), but for `LinCombGaussianWeights` the
weight emulator must have 1-D output. So we create a new regressor with
`output_shape=(3,)` -- 3 weight outputs -- and then map those to a
5-dimensional output space via a `(5, 3)` matrix.

In [37]:
# A simple weight emulator: scalar input, 3-D weight output
# Feature map produces (*eb, n, 3, 2) for multi-output
def weight_features(X):
    """Feature map for a 3-output, 2-weight regressor."""
    x = X[..., 0]  # (*eb, n)
    # For each input, produce a (3, 2) feature matrix
    ones = jnp.ones_like(x)
    # Output 0: [1, x]
    # Output 1: [1, x^2]
    # Output 2: [x, x^2]
    row0 = jnp.stack([ones, x], axis=-1)      # (*eb, n, 2)
    row1 = jnp.stack([ones, x ** 2], axis=-1)
    row2 = jnp.stack([x, x ** 2], axis=-1)
    return jnp.stack([row0, row1, row2], axis=-2)  # (*eb, n, 3, 2)


w_weights = MultivariateNormal(
    loc=jnp.array([0.5, -0.2]),
    cov=0.05 * jnp.eye(2),
)

weight_emulator = LinearGaussianRegressor(
    feature_map=weight_features,
    weights=w_weights,
    input_shape=(1,),
    output_shape=(3,),  # 3-D weight output
)

print(f"Weight emulator: {weight_emulator}")
print(f"  output_shape: {weight_emulator.output_shape}")

Weight emulator: LinearGaussianRegressor(input_shape=(1,), output_shape=(3,))
  output_shape: (3,)


In [38]:
# Linear map from 3-D weight space to 5-D output space
phi_matrix = jax.random.normal(jax.random.PRNGKey(42), (5, 3))
bias = jnp.zeros(5)

lincomb = LinCombGaussianWeights(
    weight_emulator=weight_emulator,
    phi=phi_matrix,
    bias=bias,
)

print(f"LinCombGaussianWeights: {lincomb}")
print(f"  input_shape:  {lincomb.input_shape}")
print(f"  output_shape: {lincomb.output_shape}")
print(f"  supports_joint_inputs:  {lincomb.supports_joint_inputs}")
print(f"  supports_joint_outputs: {lincomb.supports_joint_outputs}")

LinCombGaussianWeights: LinCombGaussianWeights(input_shape=(1,), output_shape=(5,))
  input_shape:  (1,)
  output_shape: (5,)
  supports_joint_inputs:  True
  supports_joint_outputs: True


### 5.2 Predictions with LinCombGaussianWeights

With `output_shape=(5,)` and `supports_joint_outputs=True`, we can now
demonstrate all four joint modes.

In [40]:
X_lc = jnp.linspace(-1, 1, 4).reshape(-1, 1)  # (4, 1)

# Mode 1: fully marginal
d1 = lincomb(X_lc)
print("Mode 1: joint_inputs=False, joint_outputs=False")
print(f" em(X): {d1}")

# Mode 2: joint inputs only
d2 = lincomb(X_lc, joint_inputs=True)
print("\nMode 2: joint_inputs=True, joint_outputs=False")
print(f" em(X): {d2}")

# Mode 3: joint outputs only
d3 = lincomb(X_lc, joint_outputs=True)
print("\nMode 3: joint_inputs=False, joint_outputs=True")
print(f" em(X): {d3}")

# Mode 4: fully joint
d4 = lincomb(X_lc, joint_inputs=True, joint_outputs=True)
print("\nMode 4: joint_inputs=True, joint_outputs=True")
print(f" em(X): {d4}")
print(f"  cov shape: {d4.cov.shape}  # (n*d_out, n*d_out) = (4*5, 4*5)")

Mode 1: joint_inputs=False, joint_outputs=False
 em(X): Normal(event_shape=(), batch_shape=(4, 5))

Mode 2: joint_inputs=True, joint_outputs=False
 em(X): MultivariateNormal(event_shape=(4,), batch_shape=(5,))

Mode 3: joint_inputs=False, joint_outputs=True
 em(X): MultivariateNormal(event_shape=(5,), batch_shape=(4,))

Mode 4: joint_inputs=True, joint_outputs=True
 em(X): MultivariateNormal(event_shape=(20,))
  cov shape: (20, 20)  # (n*d_out, n*d_out) = (4*5, 4*5)


## 6. Trajectory Sampling with `sample_trajectory`

The `sample_trajectory` method draws random **function realizations** from
an emulator. Unlike `.sample()` on the predictive distribution (which draws
independent sets of output values), trajectory sampling produces functions
that are **consistent** across different input locations: evaluating the
returned callable at different `X` values yields values from the *same*
underlying random function.

For `LinearGaussianRegressor`, this works by drawing weight vectors once
and evaluating `bias + Phi(X) @ w` at any input.

### 6.1 Drawing Trajectories

In [41]:
# Draw 3 function realizations from the polynomial regressor
g = regressor.sample_trajectory(jax.random.PRNGKey(99), n_trajectories=3)

X_traj = jnp.linspace(-2, 2, 10).reshape(-1, 1)  # (10, 1)
Y_traj = g(X_traj)

print(f"X_traj.shape: {X_traj.shape}")
print(f"Y_traj.shape: {Y_traj.shape}  # (n_trajectories, n)")
print(f"\nTrajectory 0: {Y_traj[0]}")
print(f"Trajectory 1: {Y_traj[1]}")
print(f"Trajectory 2: {Y_traj[2]}")

X_traj.shape: (10, 1)
Y_traj.shape: (3, 10)  # (n_trajectories, n)

Trajectory 0: [-4.3195395  -1.8317274  -0.18762799  0.79295206  1.2902049   1.4843237
  1.5555012   1.6839304   2.049804    2.833315  ]
Trajectory 1: [0.9853128  0.5177409  0.39969805 0.5514573  0.8932911  1.3454729
 1.8282752  2.261971   2.5668335  2.6631348 ]
Trajectory 2: [ 2.7816594   1.5858724   1.0167484   0.84786355  0.8527941   0.8051161
  0.47840568 -0.35376108 -1.9178078  -4.440159  ]


### 6.2 Consistency Across Calls

The key property of trajectory sampling is that evaluating the callable
at a subset of points returns the same values as evaluating at the full
set. This is because the weight vectors are fixed at sampling time.

In [42]:
# Evaluate the same trajectories at just the first 3 points
Y_subset = g(X_traj[:3])
print(f"Full evaluation (first 3 points): {Y_traj[:, :3]}")
print(f"Subset evaluation:                {Y_subset}")
print(f"\nValues match: {jnp.allclose(Y_traj[:, :3], Y_subset)}")

Full evaluation (first 3 points): [[-4.3195395  -1.8317274  -0.18762799]
 [ 0.9853128   0.5177409   0.39969805]
 [ 2.7816594   1.5858724   1.0167484 ]]
Subset evaluation:                [[-4.3195395  -1.8317273  -0.187628  ]
 [ 0.9853128   0.51774096  0.39969805]
 [ 2.7816594   1.5858724   1.0167484 ]]

Values match: True


### 6.3 Extra Batch Dimensions in Trajectories

Trajectory callables also handle extra batch dimensions.

In [43]:
X_traj_batch = jnp.stack([X_traj, X_traj + 0.5])  # (2, 10, 1)
Y_traj_batch = g(X_traj_batch)
print(f"X_traj_batch.shape: {X_traj_batch.shape}")
print(f"Y_traj_batch.shape: {Y_traj_batch.shape}")
print(f"  (n_trajectories=3, extra_batch=2, n=10)")

X_traj_batch.shape: (2, 10, 1)
Y_traj_batch.shape: (3, 2, 10)
  (n_trajectories=3, extra_batch=2, n=10)


## Summary

Key points from this notebook:

1. **`Emulator`** is the abstract base class. It defines the `__call__` interface,
   the four joint modes, and the shape contract.

2. **`GaussianEmulator`** specializes this to Gaussian predictives. Subclass it
   and implement `predict_mean`, `predict_variance`, and optionally
   `predict_covariance`.

3. **Custom emulators** are easy to build. The `ToyGP` example shows a GP
   with `supports_joint_inputs=True`; the `SyntheticLikelihoodEmulator`
   shows an independent-prediction emulator with
   `supports_joint_inputs=False`.

4. **`LinearGaussianRegressor`** provides analytical Gaussian predictions
   via a feature map and weight distribution. It supports trajectory
   sampling via weight-space sampling.

5. **`LinCombGaussianWeights`** linearly transforms a weight emulator to a
   higher-dimensional output space.

6. **`sample_trajectory`** draws consistent function realizations that
   can be evaluated at any input locations.